# Try Haze

[Haze](https://github.com/ritabratamaiti/haze-lang) is a small language for checking software that
has to cope with things going wrong some of the time. You write a model of a system together with
the parts you don't control, such as lost messages or a flaky service, state what the system
should guarantee, and Haze checks it with exact probabilities. When a guarantee can fail, it shows
you a run where it does.

Run the cells from top to bottom (in Colab, **Runtime → Run all**). The first cell installs Haze,
which takes a few seconds. After that, every cell that starts with `%%haze` is a Haze model, and
running it checks the model. You can edit any of them and run them again. Each cell is checked on
its own, so a cell can't use declarations from another one.

In [1]:
# Installs Haze if it isn't installed yet, and defines %%haze, which checks the model in a cell.
import os, pathlib, shutil, subprocess, sys, tempfile
from IPython.core.magic import register_cell_magic

local_bin = pathlib.Path.home() / ".local" / "bin"
os.environ["PATH"] = f"{local_bin}{os.pathsep}{os.environ['PATH']}"
if shutil.which("haze") is None:
    if sys.platform == "win32":
        raise RuntimeError("Install Haze first: https://github.com/ritabratamaiti/haze-lang#install")
    subprocess.run("curl -sSfL https://raw.githubusercontent.com/ritabratamaiti/haze-lang/main/install.sh | sh",
                   shell=True, check=True)
print(subprocess.run(["haze", "--version"], capture_output=True, text=True).stdout.strip(), "is ready")

@register_cell_magic
def haze(line, cell):
    """Checks the Haze model in this cell. Options after %%haze, such as --json, go to haze."""
    with tempfile.NamedTemporaryFile("w", suffix=".hz", delete=False, encoding="utf-8") as f:
        f.write(cell)
    try:
        r = subprocess.run(["haze", *line.split(), f.name], capture_output=True, text=True, encoding="utf-8")
        print((r.stdout + r.stderr).replace(f.name, "cell").rstrip())
    finally:
        os.unlink(f.name)

haze 0.1.0 is ready


## Exact probabilities

Random values are written with `~`. `#prob` gives the probability that something holds, and
`#eval` the whole distribution. The answers are exact fractions, not estimates from sampling.

In [2]:
%%haze
input die1 : Num ~ uniform(1, 6)
input die2 : Num ~ uniform(1, 6)
#prob die1 + die2 == 7
#eval max(die1, die2)

cell:3:1: info: #prob die1 + die2 == 7
  probability 0.166667 (=1/6)
cell:4:1: info: #eval max(die1, die2)
  6  ↦ 0.305556 (=11/36)
  5  ↦ 0.25
  4  ↦ 0.194444 (=7/36)
  3  ↦ 0.138889 (=5/36)
  2  ↦ 0.083333 (=1/12)
  1  ↦ 0.027778 (=1/36)


## A payment client that retries

Here is a more realistic model. A client sends a payment request; if the request or the reply is
lost, it waits for a timeout and tries again, up to three times. The server charges the card every
time a request reaches it.

We only know the loss rate roughly ("around 2%, maybe 5%"), so `loss` is a *fuzzy parameter*: each
value comes with how plausible it is. Answers that depend on it are a range, from *belief* (how
strongly what we know guarantees the property) to *plausibility* (how strongly it allows it).

A `claim` asks how likely something is, and a `theorem` asks Haze to prove it always holds. Haze
accepts the first claim and rejects the theorem, with the run that breaks it: the first request
got through, its reply was lost, and the retry charged the card a second time.

In [3]:
%%haze
enum Phase = Sending | Waiting | Done | GaveUp
param loss : Num ~= {0.01: 0.6, 0.02: 1, 0.05: 0.5}

system Client {
  var phase : Phase := Sending
  var attempts : Num := 0
  var charges : Num := 0
  step {
    match phase {
      Sending => {
        attempts := attempts + 1
        if flip(loss) { phase := Waiting }           -- the request is lost
        else {
          charges := charges + 1                     -- the server charges the card
          if flip(loss) { phase := Waiting }         -- the reply is lost
          else { phase := Done }
        }
      }
      Waiting => { if attempts < 3 { phase := Sending } else { phase := GaveUp } }
      _ => skip
    }
  }
}

claim delivered : Client |= eventually[6] phase == Done with prob >= 0.99
theorem never_charged_twice : Client |= always[6] charges <= 1

cell:25:1: info: claim delivered
  belief ≈0.999506 · plausibility ≈0.99997 — graded
  grade prob+fuzzy · 3 parameter assignment(s) · ≤1 states/step
  bound holds: belief ≥ 0.99
  by fuzzy parameter (π = possibility):
    loss = 0.01  π 0.6     ≈0.999992
    loss = 0.02  π 1       ≈0.999938
    loss = 0.05  π 0.5     ≈0.999073
cell:26:1: error: theorem never_charged_twice
  belief ≈0.965193 · plausibility ≈0.986011 — graded
  grade prob+fuzzy · 3 parameter assignment(s) · ≤5 states/step
  theorem rejected: belief is ≈0.965193, not 1
  by fuzzy parameter (π = possibility):
    loss = 0.01  π 0.6     ≈0.990003
    loss = 0.02  π 1       ≈0.980024
    loss = 0.05  π 0.5     0.950363 (=76029/80000)
  counterexample at step 3 (path mass ≈9.801e-5 (=9801/100000000), degree 0):
    parameters: loss = 0.01
    state: phase = Waiting, attempts = 2, charges = 2


The fix is an *idempotency key*: the server remembers each payment it has charged, so a retry
gets the stored result instead of a new charge. Change the line `charges := charges + 1` above to
`if charges == 0 { charges := 1 }` and run the cell again. Haze now proves the theorem for every
loss rate in the model.

## Runs of any length

Without a bound, `eventually` looks at the whole run, however long it lasts. In craps, the shooter
keeps rolling until the point comes up again or a 7 does, which can take any number of rolls. Haze
works out the exact chance of winning, which is the well-known 244/495.

In [4]:
%%haze
enum Phase = ComeOut | OnPoint | Won | Lost
system Craps {
  var phase : Phase := ComeOut
  var point : Num := 0
  step {
    let roll := uniform(1, 6) + uniform(1, 6)
    match phase {
      ComeOut => {
        if roll == 7 or roll == 11 { phase := Won }
        else if roll == 2 or roll == 3 or roll == 12 { phase := Lost }
        else { point := roll; phase := OnPoint }
      }
      OnPoint => { if roll == point { phase := Won } else if roll == 7 { phase := Lost } }
      _ => skip
    }
  }
}
claim shooter_wins : Craps |= eventually phase == Won
#expect Craps |= steps until phase == Won or phase == Lost

cell:18:1: info: claim shooter_wins
  probability 0.492929 (=244/495) — graded
  grade prob · 1 parameter assignment(s) · 21 reachable states
cell:19:1: info: #expect Craps |= steps until phase == Won or phase == Lost
  expected steps 3.375758 (=557/165)


## Your turn

This cell has a background job that calls a flaky service and retries with a backoff. Try changing
the failure rate, the number of retries or the property, and run it again. For example, how many
retries does it take to finish 99.9% of the time?

To go further, read the [tutorial](https://github.com/ritabratamaiti/haze-lang/blob/main/examples/Tutorial/JobWorker.hz),
the [language reference](https://github.com/ritabratamaiti/haze-lang/blob/main/docs/Language-Reference.md)
and the [examples](https://github.com/ritabratamaiti/haze-lang/tree/main/examples).

In [5]:
%%haze
param fails : Num ~= {0.1: 1, 0.3: 0.5}

system Job {
  var tries : Num := 0
  var done : Bool := false
  var waited : Num := 0
  step {
    if not done and tries < 4 {
      tries := tries + 1
      if flip(fails) { waited := waited + 100 * tries } else { done := true }
    }
  }
}

claim finishes : Job |= eventually done with prob >= 0.99
#expect Job |= after[4] waited

cell:15:1: info: claim finishes
  belief 0.9959 · plausibility 0.9999 — graded
  grade prob+fuzzy · 2 parameter assignment(s) · 9 reachable states
  bound holds: belief ≥ 0.99
  by fuzzy parameter (π = possibility):
    fails = 0.1  π 1       0.9999
    fails = 0.3  π 0.5     0.9919
cell:16:1: info: #expect Job |= after[4] waited
  lower expectation 12.34 · upper expectation 35.84
  by fuzzy parameter (π = possibility):
    fails = 0.1  π 1       E 12.34
    fails = 0.3  π 0.5     E 59.34
